<a href="https://colab.research.google.com/github/Ammaryasir4673/My_Python_Projects/blob/main/%09Materials_Science/CGCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Practical exercise: Convert a crystal structure into a graph using pymatgen

Mounted at /content/drive


In [ ]:
#Step 1. Install and import libraries

In [ ]:
!pip install pymatgen
from pymatgen.core import Lattice, Structure
import numpy as np
import pandas as pd

In [ ]:
#Step 2. Create a sample crystal structure

In [ ]:
lattice = Lattice.cubic(4.33)

species = ["Ti", "Ti", "Ti", "Ti",
           "C", "C", "C", "C"]

coords = [
    [0, 0, 0],
    [0, 0.5, 0.5],
    [0.5, 0, 0.5],
    [0.5, 0.5, 0],

    [0.5, 0, 0],
    [0, 0.5, 0],
    [0, 0, 0.5],
    [0.5, 0.5, 0.5]
]

structure = Structure(lattice, species, coords)

In [ ]:
#Step 3. Save it as a CIF file and read it again

In [ ]:
structure.to(filename="TiC_demo.cif")

structure = Structure.from_file("TiC_demo.cif")

In [ ]:
#n real projects, you will generally start with an existing CIF file rather than creating the structure manually.

In [ ]:
#Step 4. Extract atomic information

In [ ]:
print("Number of atoms:", len(structure))

print("Composition:", structure.composition)

print("Lattice:")
print(structure.lattice)

In [ ]:
#Now extract information about each atom:
for i, site in enumerate(structure):

    print("Atom index:", i)
    print("Element:", site.species_string)
    print("Atomic number:", site.specie.number)
    print("Fractional coordinates:", site.frac_coords)
    print("Cartesian coordinates:", site.coords)

    print("---------------------")

In [ ]:
#Step 5. Find neighboring atoms

In [ ]:
neighbors = structure.get_all_neighbors(r=2.6)

In [ ]:
#Now print the neighboring atoms:
for i, atom_neighbors in enumerate(neighbors):

    print("Central atom:", i, structure[i].species_string)

    for neighbor in atom_neighbors:

        print(
            "Neighbor index:", neighbor.index,
            "Element:", neighbor.species_string,
            "Distance:", round(neighbor.nn_distance, 3)
        )

    print("-----------------------")

In [ ]:
#Step 6. Convert atoms into nodes

In [ ]:
node_features = []

for site in structure:

    node_features.append([
        site.specie.number
    ])

node_features = np.array(node_features)

print(node_features)

In [ ]:
#Step 7. Convert neighboring atoms into edges

In [ ]:
edge_sources = []
edge_targets = []
edge_distances = []

for i, atom_neighbors in enumerate(neighbors):

    for neighbor in atom_neighbors:

        edge_sources.append(i)
        edge_targets.append(neighbor.index)
        edge_distances.append(neighbor.nn_distance)

In [ ]:
#Let's convert the arrays:
edge_sources = np.array(edge_sources)
edge_targets = np.array(edge_targets)
edge_distances = np.array(edge_distances)

print("Source nodes:", edge_sources)
print("Target nodes:", edge_targets)
print("Distances:", edge_distances)

In [ ]:
#Step 8. Combine everything into one graph

In [ ]:
graph = {
    "node_features": node_features,
    "edge_sources": edge_sources,
    "edge_targets": edge_targets,
    "edge_distances": edge_distances
}

print("Node features shape:", graph["node_features"].shape)

print("Number of edges:", len(graph["edge_distances"]))

In [ ]:
#Step 9. Convert the graph into PyTorch tensors

In [ ]:
import torch

node_tensor = torch.tensor(
    node_features, dtype=torch.float32
)

edge_index = torch.tensor(
    np.array([edge_sources, edge_targets]),
    dtype=torch.long
)

edge_distance_tensor = torch.tensor(
    edge_distances, dtype=torch.float32
)

In [ ]:
#Check their shapes:
print("Node tensor:", node_tensor.shape)
print("Edge index:", edge_index.shape)
print("Edge distances:", edge_distance_tensor.shape)

In [ ]:
#Step 10. Convert distances into Gaussian features

In [ ]:
dmin = 0
dmax = 2.6
step = 0.2

centers = np.arange(dmin, dmax + step, step)

gaussian_features = np.exp(
    -((edge_distances[:, None] - centers[None, :]) ** 2)
    / (step ** 2)
)

print("Gaussian features shape:", gaussian_features.shape)

In [ ]:
#Now we have t0 pass the graph information into the actual CGCNN neural network.

In [ ]:
'''Complete workflow at a glance
Here is the entire process from the beginning:
1. CIF file

   Contains crystal structure information.
2. Pymatgen

   Reads atomic structure and coordinates.
3. Neighbor identification

   Finds neighboring atoms and distances.
4. Graph construction

   Creates nodes and edges.
5. Feature conversion

   Converts atomic and distance information into numerical features.
6. PyTorch and CGCNN

   Processes graphs through graph convolution, pooling and neural network layers.
7. Prediction

   Predicts the target material property.
8. Loss calculation

   Compares predictions with actual DFT values.
9. Backpropagation and optimization

   Updates neural network weights.
10. Validation

    Monitors model performance during training.
11. Testing

    Evaluates performance on unseen structures.
12. New material prediction

    Uses the trained model to predict properties of new crystals.'''